## Import

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

## Questions :

- What is the most sell product ?
- What is the less sell product ?
- What is the most sell aisle ?
- What is the less sell aisle ?
- What is the most important client ?
- Which day has the most orders ?
- Which hour of day has the most orders ? 
- When is the busiest time for orders ?
- Which products are most often add in first in cart ?
- Which products are most ofthen reordered ?
- Which products have the most probability to reorder ?
- How many client per day ?
- How many of each product are sold per day ?
- For Each Client : Why Client buy , When and How many ?
- For Each Clients : What is his favorite product ?
- For Each Client : What is his favorite moment ?
- What is the top10 products sales per day ?

## Data Traitment

### Import Pandas and datasets

In [ ]:
aisles = pd.read_csv("aisles.csv")
departments = pd.read_csv("departments.csv")
order_products = pd.read_csv("order_products.csv")
orders = pd.read_csv("orders.csv")
products = pd.read_csv("products.csv")

### Print all datasets infos

In [ ]:
aisles.info()
departments.info()
order_products.info()
orders.info()
products.info()

### Merge all datasets in df

In [ ]:
df = pd.merge(order_products,orders,on="order_id")
df = pd.merge(df,products,on="product_id")
df = pd.merge(df,departments,on="department_id")
df = pd.merge(df,aisles,on="aisle_id")

In [ ]:
df.info()

### Define the right column types

In [ ]:
# fillna is for replace Nan value to -1
df["product_id"] = df["product_id"].astype(int)
df['reordered'] = df['reordered'].fillna(-1).astype(int)
df['user_id'] = df['user_id'].fillna(-1).astype(int)
df['add_to_cart_order'] = df['add_to_cart_order'].fillna(-1).astype(int)
df['order_number'] = df['order_number'].fillna(-1).astype(int)
df["days_since_prior_order"] = df["days_since_prior_order"].fillna(-1).astype(int)

df['order_hour_of_day'] = df['order_hour_of_day'].fillna(-1).astype(int)
# remove hour superior of 23 and inferior of 0
df = df[(df["order_hour_of_day"] < 24) & (df["order_hour_of_day"] > -1)]

df['order_dow'] = df['order_dow'].fillna(-1).astype(int)
df = df[(df["order_dow"] < 7) & (df["order_dow"] > -1)]


In [ ]:
df.head(5)

In [ ]:
df.isnull().sum()

## Answer to the question

### What is the most sell product ?

In [ ]:
# barplot of the top10 of most sell products
df["product_name"].value_counts().head(10).plot(kind="bar")

Thanks to the graph, we can see the ten best-selling products and the number of sales for each product.

The three best-sellers are bananas, organic banana bags, and organic bananas. 

This information can help us make placement decisions. We could place the best-selling items at the back of the store so that customers are more tempted to buy the products they pass by.

### What is the less sell product ?

In [ ]:
# barplot of the top10 of less sell products
df["product_name"].value_counts().tail(10).plot(kind="bar")

Thanks to this graph, we can see the best-selling products.

This information can help us choose where to place products. We could place the least-sold products next to the best-selling ones to improve their sales.

### What is the most sell aisle ?

In [ ]:
# barplot of the top10 sell aisles
df["aisle"].value_counts().head(10).plot(kind="bar")

### What is the less sell aisle ?

In [ ]:
df["aisle"].value_counts().tail(10).plot(kind="bar")

This graph shows the aisles with the highest sales. 

This information could help us decide where to place products. We could place the aisles with the highest sales at the back of the store so that customers are more likely to walk past the products.

### What is the most important client ?

In [ ]:
# barplot of the top10 most imporant clients
df["user_id"].value_counts().head(10).plot(kind="bar")

This graph allows us to see which customers place the most orders. 

This information could help us build customer loyalty. Depending on the number of orders placed, we could offer discounts.

### Which day has the most orders ?

In [ ]:
# graph of the number of orders per day
# 0 : Sunday, 1 : Monday, 2 : Tuesday, 3 : Wednesday, 4 : Thursday, 5 : Friday, 6 : Saturday, -1 : Nan/No value
df["order_dow"].value_counts().plot(kind="bar")

Thanks to this graph, we can see which day has the most orders.

This information can help us allocate employees throughout the week.

### Which hour of day has the most orders ? 

In [ ]:
# graph of the number of orders per hour
df["order_hour_of_day"].value_counts().plot(kind="bar")

Thanks to this graph, we can see which day has the most orders. 

This information can help us allocate employees throughout the day.

### When is the busiest time for orders ?

In [ ]:
# create a new column with day and hour
# format : day-hour
df["order_dow_hour"] = df["order_dow"].astype(str) +"-"+df["order_hour_of_day"].astype(str)

In [ ]:
# graph showing the top 10 of the day and the hours with the most orders
# 0 : Sunday, 1 : Monday, 2 : Tuesday, 3 : Wednesday, 4 : Thursday, 5 : Friday, 6 : Saturday, -1 : Nan/No value
df["order_dow_hour"].value_counts().head(10).plot(kind="bar")

The graph shows us which day has the most orders.

This information can help us with the distribution of employees throughout the week and the hours.

### Which products are most often add in first in cart ?

In [ ]:
#top 10 product more ofthen add in first
df_temp = df[(df["add_to_cart_order"] == 1)]
df_temp["product_name"].value_counts().head(10)

This graph allows us to see which products are added to the cart first.

This information can help us decide where to place products. We can encourage customers to buy products that are selling less well by placing them next to them.

### Which products are most ofthen reordered ?

In [ ]:
#top 10 product more ofthen add in first
df_temp = df[(df["reordered"] == 1)]
df_temp["product_name"].value_counts().head(10)

This graph allows us to see which products are repurchased most often

This information can help us decide where to place products. We can encourage customers to buy products that sell less well by placing them next to them.

### Which products have the most probability to reorder ?

In [ ]:

def get_proba_product_reordered(give_df):
    df_temp = give_df[(give_df["reordered"] == 1)] #keep all product with reordered = 1
    
    dict_count_product_reordered = dict(df_temp["product_name"].value_counts()) #for all products count how many product have reordered = 1
    dict_count_product = dict(give_df["product_name"].value_counts()) #count all products

    dict_proba_product_reordered = {
        "product_name":[],
        "proba_reordered":[]
    } 

    for key,value in dict_count_product_reordered.items():
        count_product_reordered = value
        count_product = dict_count_product[key]
        if count_product and count_product_reordered and count_product > 0 and count_product_reordered > 0 :
            dict_proba_product_reordered["product_name"].append(key)
            dict_proba_product_reordered["proba_reordered"].append(count_product_reordered/count_product)
        else :
            dict_proba_product_reordered["product_name"].append(key)
            dict_proba_product_reordered["proba_reordered"].append(0)

    df_proba_product_reordered = pd.DataFrame(dict_proba_product_reordered)
    return df_proba_product_reordered

get_proba_product_reordered(df).head(10)

This graph shows the probability that an item will be repurchased.

This information can be used to estimate customer loyalty, satisfaction, and needs. 

### How many client per day ?

In [ ]:
def get_daily_count_client(give_df):
    dict_count_client = {}
    for i in range(7):
        nb_client = len(give_df[give_df["order_dow"] == i]["user_id"].unique())
        dict_count_client[i] = nb_client
    
    return dict_count_client

get_daily_count_client(df)

This graph shows how many customers we have per day and therefore how many people we can expect to visit the store.

### How many of each product are sold per day ?

In [ ]:
def get_daily_product_sales(give_df):
    dict_count_daily_product_sales = {}
    for i in range(7):
        df_temp = give_df[give_df["order_dow"] == i]
        dict_count_daily_product_sales[i] = df_temp["product_name"].value_counts().astype(int).to_dict()
    
    return dict_count_daily_product_sales

with open('count_daily_product_sales.json', 'w') as file:
    json.dump(get_daily_product_sales(df), file, indent=4)

This graph shows how many products are sold per day and therefore how much stock should be expected.

### For Each Client : Why Client buy , When and How many ?

In [ ]:
def get_user_habit(give_df):
    habit_per_client = {}
    for index,row in give_df.iterrows():
        if index < 10000000000 :
            user_id = row["user_id"]
            order_id = row["order_id"]
            date = row["order_dow_hour"]
            day = row["order_dow"]
            hour = row["order_hour_of_day"]
            product = row["product_name"]
            if user_id not in habit_per_client:
                habit_per_client[user_id] = {
                    "count":0,
                    "day": {0:0,1:0,2:0,3:0,4:0,5:0,6:0,"Nan":0},
                    "hour":{0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:0,14:0,15:0,16:0,17:0,18:0,19:0,20:0,21:0,22:0,23:0,"Nan":0}
                }
                habit_per_client[user_id][order_id] = {
                    "date":date,
                    "content":[product]
                }
                habit_per_client[user_id]["count"] += 1
                if day < 0 : 
                    habit_per_client[user_id]["day"]["Nan"] += 1
                else : 
                    habit_per_client[user_id]["day"][day] += 1
                if hour < 0 :
                    habit_per_client[user_id]["hour"]["Nan"] += 1
                else :
                    habit_per_client[user_id]["hour"][hour] += 1

            else :
                if order_id in habit_per_client[user_id]:
                    habit_per_client[user_id][order_id]["content"].append(product)
                else :
                    habit_per_client[user_id][order_id] = {
                        "date":date,
                        "content":[product]
                    }
                    habit_per_client[user_id]["count"] += 1
                    if day < 0 : 
                        habit_per_client[user_id]["day"]["Nan"] += 1
                    else : 
                        habit_per_client[user_id]["day"][day] += 1
                    if hour < 0 :
                        habit_per_client[user_id]["hour"]["Nan"] += 1
                    else :
                        habit_per_client[user_id]["hour"][hour] += 1


    sorted_habit_per_client = dict(sorted(habit_per_client.items(), key=lambda item: len(item[1]), reverse=True))
    return sorted_habit_per_client

#with open('habit.json', 'w') as file:
    #json.dump(get_user_habit(df), file, indent=4)

This data allows us to understand customer habits, their preferences, when they place their orders, and the number of orders they have placed.

### For Each Clients : What is his favorite product ?

In [ ]:

def get_client_favorite_product(habit:dict):
    dict_client_favorite_product = {
        "user_id":[],
        "product_name":[]
    }

    for client in habit.keys():
        user_id = client
        count_order = habit[client]["count"]
        if count_order > 0 :
            dict_count_client_product = {}
            for order in habit[client].keys(): 
                if order != "hour" and order != "day" and order != "count":
                    for product in habit[client][order]["content"]:
                        if product not in dict_count_client_product:
                            dict_count_client_product[product] = 1
                        else :
                            dict_count_client_product[product] += 1
            dict_client_favorite_product["user_id"].append(user_id)
            dict_client_favorite_product["product_name"].append(max(dict_count_client_product,key=dict_count_client_product.get))
    
    return pd.DataFrame(dict_client_favorite_product)


with open('habit.json', 'r') as file:
    client_habits = json.load(file)

get_client_favorite_product(client_habits)

This data allows us to identify each customer's favorite product. By taking the favorite product of the majority of customers, we can choose which promotion to run. 

### For Each Client : What is his favorite moment ?

In [ ]:
def get_client_favorite_moment(habit:dict):
    dict_client_favorite_moment = {
        "user_id":[],
        "favorite_day":[],
        "favorite_hour":[]
    }
    for client in habit.keys():
        user_id = client
        count_order = habit[client]["count"]
        if count_order > 0 :
            favorite_day = max(habit[client]["day"],key=habit[client]["day"].get)
            favorite_hour = max(habit[client]["hour"],key=habit[client]["hour"].get)
            
            dict_client_favorite_moment["user_id"].append(user_id)
            dict_client_favorite_moment["favorite_day"].append(favorite_day)
            dict_client_favorite_moment["favorite_hour"].append(favorite_hour)

    return pd.DataFrame(dict_client_favorite_moment)

with open('habit.json', 'r') as file:
    client_habits = json.load(file)

get_client_favorite_moment(client_habits)


This data allows us to know when customers prefer to visit. Taking into account the preferred time of the majority of customers can help us choose when to run limited-time promotions.

### Top10 products sales per day

In [ ]:
def get_mosts_sell_per_day(count_daily_product_sales:dict):
    dict = {}
    for i in range(7):
        day = count_daily_product_sales.get(str(i))
        top10 = list(day.keys())[:10]
        dict[i] = top10 
    return dict           


with open('count_daily_product_sales.json', 'r') as file:
    cdps = json.load(file)

get_mosts_sell_per_day(cdps)

This data allows you to see the best-selling items per day, which could help you choose which products to promote in weekly promotions.

## Machine Training

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
def get_last_order(user_id,give_df):
    return give_df[give_df["user_id"] == user_id]["order_id"].max()

def build_data_from_dataset(user_id:int,product_name:str,give_df,order_id=None):

    if not order_id :
        order_id = get_last_order(user_id,give_df)

    user_data = give_df[give_df["user_id"] == user_id]

    user_product_count = len(user_data[user_data["product_name"]==product_name])

    user_order_count = user_data["order_number"].max()

    if user_order_count > 0 :
        product_frequency = user_product_count/user_order_count
    else :
        product_frequency = 0

    product_sell = len(give_df[give_df["product_name"]==product_name])

    order_data = df[df["order_id"] == order_id].iloc[0]
    order_dow = order_data["order_dow"]
    order_hour = order_data["order_hour_of_day"]
    aisle_id = order_data["aisle_id"]
    department_id = order_data["department_id"]

    return [
        user_product_count,
        user_order_count,
        product_frequency,
        product_sell,
        order_dow,
        order_hour,
        aisle_id,
        department_id
    ]


def get_count_client_buy_product(give_df):
    dict_count_client_buy_product = {}
    for index,row in give_df.iterrows():
        user_id = row["user_id"]
        product_name = row["product_name"]
        if user_id not in dict_count_client_buy_product :
            dict_count_client_buy_product[user_id] = {
                product_name:1
            }
        else : 
            if product_name not in dict_count_client_buy_product[user_id]:
                dict_count_client_buy_product[user_id][product_name] = 1
            else :
                dict_count_client_buy_product[user_id][product_name] += 1
    
    with open('count_client_buy_product.json', 'w') as file:
        json.dump(dict_count_client_buy_product, file, indent=4)
    return True

def create_count_client_buy_product(json_data,give_df):
    give_df["client_product_count"] = None
    for index,row in give_df.iterrows():
        user_id = str(row["user_id"])
        product_name = row["product_name"]
        give_df.loc[index, "client_product_count"] = json_data[user_id][product_name]

def create_client_total_order(give_df,habit:dict):
    give_df['client_total_orders'] = None
    for index,row in give_df.iterrows():
        user_id = str(row["user_id"])
        give_df.loc[index, "client_total_orders"] = habit[user_id]["count"]


def create_client_product_frequency(give_df):
    give_df["client_product_frequency"] = None
    for index,row in give_df.iterrows():
        client_product_count = row["client_product_count"]
        client_total_orders = row["client_total_orders"]
        if client_total_orders and client_product_count and client_total_orders > 0 : 
            frequency = client_product_count/client_total_orders
        else :
            frequency = 0
        give_df.loc[index, "client_product_frequency"] = frequency


def get_product_popularity(give_df):
    dict_product_popularity = give_df["product_name"].value_counts().astype(int).to_dict()
    with open('product_popularity.json', 'w') as file:
        json.dump(dict_product_popularity, file, indent=4)
    return True
    
def create_product_popularity(give_df,json_data):
    give_df["product_popularity"] = None
    for index,row in give_df.iterrows():
        product_name = row["product_name"]
        give_df.loc[index, "product_popularity"] = json_data[product_name]



with open('count_client_buy_product.json', 'r') as file:
    ccbp = json.load(file)

with open('habit.json', 'r') as file:
    client_habits = json.load(file)

create_count_client_buy_product(ccbp,df)
create_client_total_order(df,client_habits)
create_client_product_frequency(df)
get_product_popularity(df)

with open('product_popularity.json', 'r') as file:
    pp = json.load(file)

create_product_popularity(df,pp)

df.head(10)
df.to_csv("dataset_for_ML1.csv",index="False")


In [ ]:
features = [
    "user_product_frequency",
    "user_product_count",
    "product_popularity",
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "aisle_id",
    "department_id"
]

X = df[features]
y = df["reordered"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    random_state=42
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print(classification_report(y_test, pred))


def predict_reorder(user_id, product_name, give_df, model, order_id=None):

    X_pred = build_data_from_dataset(
        user_id,
        product_name,
        give_df,
        order_id
    )

    prediction = model.predict(X_pred)[0]
    probability = model.predict_proba(X_pred)[0][1]

    return {
        "prediction": int(prediction),
        "probability": float(probability)
    }



In [ ]:
def get_top10_items_per_aisle(): 

    dic = {}
    for aisle in df["aisle"].unique():
        df_temp = df[(aisle == df["aisle"])]
        newdf = df_temp["product_name"].value_counts().head(10)
        dic[aisle] = newdf.to_dict()
        
    
    with open('top10.json', 'w') as file:
        json.dump(dic, file, indent=4)

get_top10_items_per_aisle()

In [ ]:
#PEOPLE WHO LOVE THIS PRODUCT TEND TO ALSO BUY THESE

def get_top10_items_per_user():
    dic = {}
    for user in df["user_id"].unique():
        df_temp = df[(aisle == df["aisle"])]
        newdf = df_temp["product_name"].value_counts().head(10)
        dic[aisle] = newdf.to_dict()
        
    
    with open('top10.json', 'w') as file:
        json.dump(dic, file, indent=4)

get_top10_items_per_aisle()
    
